# SAILIR box1l training on Colab
Run cells from top to bottom. Data and checkpoints are stored on Drive so they survive runtime resets.

In [ ]:
import importlib.util, subprocess, sys
missing = [name for name, module in [('torch','torch'), ('numpy','numpy'), ('matplotlib','matplotlib'), ('pytest','pytest')] if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Dependencies ready; CUDA available:', __import__('torch').cuda.is_available())

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys
drive.mount('/content/drive')
REPO = Path('/content/SAILIR')
if not REPO.exists(): subprocess.check_call(['git','clone','https://github.com/thisisanshgupta/SAILIR.git',str(REPO)])
WORK = Path('/content/drive/MyDrive/SAILIR_box1l')
DATA = WORK/'data'; CHECKPOINTS = WORK/'checkpoints'; LOGS = WORK/'logs'
for path in (DATA,CHECKPOINTS,LOGS): path.mkdir(parents=True,exist_ok=True)
os.chdir(REPO)
print('Repository:', REPO, '| durable workspace:', WORK)

In [ ]:
print('Running unit, topology, Laporta, scramble round-trip, and triangle-box regression tests...')
subprocess.run([sys.executable,'-m','pytest','-q','tests'],cwd=REPO,check=True)
print('All tests passed; training may proceed.')

In [ ]:
import multiprocessing, time
N_WORKERS = min(4, multiprocessing.cpu_count())
N_PER_WORKER = (30000 + N_WORKERS - 1)//N_WORKERS
RAW = DATA/'box1l_training_data.jsonl'
print(f'Generating about {N_WORKERS*N_PER_WORKER:,} trajectories (5–20 IBP steps each). Runtime depends on Colab CPU allocation; the generator reports elapsed time and ETA.')
if RAW.exists():
    print('Reusing durable JSONL:', RAW)
else:
    subprocess.run([sys.executable,'scripts/data_gen/run_local_datagen.py','--topology','box1l','--n_workers',str(N_WORKERS),'--n_scrambles_per_worker',str(N_PER_WORKER),'--seed','42','--prime','1009','--output',str(RAW)],cwd=REPO,check=True)


In [ ]:
PACKED = DATA/'box1l_tensors'
subprocess.run([sys.executable,'scripts/data_gen/preprocess_to_tensors.py','--topology','box1l','--input',str(RAW),'--output_dir',str(PACKED),'--val_split','0.1','--test_split','0.1','--seed','42'],cwd=REPO,check=True)

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
train_log = LOGS/'box1l_train.log'
cmd = [sys.executable,'-u','scripts/train/train_classifier.py','--topology','box1l','--data_dir',str(PACKED),'--output_dir',str(CHECKPOINTS),'--prime','1009','--epochs','30','--batch_size','256','--lr','4e-4','--device',device,'--resume','auto']
print('Training device:',device,'| automatic resume checks latest checkpoint | checkpoints are written every epoch')
with train_log.open('a') as log: subprocess.run(cmd,cwd=REPO,stdout=log,stderr=subprocess.STDOUT,check=True)
print('Training finished; log:',train_log)

In [ ]:
best = torch.load(CHECKPOINTS/'best_model.pt',map_location='cpu',weights_only=False)
subprocess.run([sys.executable,'scripts/plots/plot_training_curve.py','--log',str(train_log),'--out',str(WORK/'box1l_training_curve.pdf'),'--best-epoch',str(best['epoch'])],cwd=REPO,check=True)
print('Best validation epoch:',best['epoch'],'| validation loss:',best['val_metrics']['loss'])

In [ ]:
import pickle
MODEL = str(CHECKPOINTS/'best_model.pt')
ONE = WORK/'one_step_box1l.pkl'
subprocess.run([sys.executable,'scripts/eval/onestep_worker.py','--integral','1,-1,1,0','--topology','box1l','--box-masters-only','--model-checkpoint',MODEL,'--output',str(ONE),'--beam_width','20','--prime','1009','--device',device],cwd=REPO,check=True)
with ONE.open('rb') as f: one = pickle.load(f)
expected = {(1,0,1,0):496}
assert one['success'] and one['final_expr'] == expected, one
print('One-step exact check passed:',one['final_expr'])

In [ ]:
sys.path.insert(0,str(REPO)); sys.path.insert(0,str(REPO/'sailir'))
from tests.box1l_laporta import solve_box
import ibp_env
ibp_env.set_topology('box1l'); ibp_env.set_prime(1009)
reference = solve_box(ibp_env)
for integral in [(1,-1,1,0),(2,-2,2,1),(2,2,1,-1)]:
    output = WORK/f'hier_{"_".join(map(str,integral))}.pkl'
    wd = WORK/f'work_{"_".join(map(str,integral))}'
    subprocess.run([sys.executable,'scripts/eval/hierarchical_reduction.py','--integral',','.join(map(str,integral)),'--topology','box1l','--box-masters-only','--local','--output',str(output),'--work-dir',str(wd),'--model-checkpoint',MODEL,'--beam_width','20','--prime','1009'],cwd=REPO,check=True)
    with output.open('rb') as f: result = pickle.load(f)
    assert result['success'] and result['final_expr'] == reference(integral), (integral,result['final_expr'],reference(integral))
    print('Laporta comparison passed:',integral,'->',result['final_expr'])